In [2]:
import pandas as pd, numpy as np, pickle, warnings
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore')
print('Imports ✓')

Imports ✓


In [4]:
X_train = pd.read_csv('data/X_train.csv')
X_test  = pd.read_csv('data/X_test.csv')
y_train = pd.read_csv('data/y_train.csv').squeeze()
y_test  = pd.read_csv('data/y_test.csv').squeeze()
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

Train: (243, 13) | Test: (61, 13)


In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_grid = GridSearchCV(LogisticRegression(max_iter=2000, random_state=42),
    {'C':[0.01,0.1,1,10,100],'solver':['lbfgs','liblinear']}, cv=cv, scoring='accuracy', n_jobs=-1)
lr_grid.fit(X_train, y_train)
best_lr = lr_grid.best_estimator_
print(f'Best params: {lr_grid.best_params_}')
print(f'CV score: {lr_grid.best_score_:.4f} | Test: {accuracy_score(y_test, best_lr.predict(X_test)):.4f}')

Best params: {'C': 0.1, 'solver': 'lbfgs'}
CV score: 0.8354 | Test: 0.8525


In [6]:
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42),
    {'n_estimators':[100,200],'max_depth':[None,5,10],'min_samples_split':[2,5]}, cv=cv, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
print(f'Best params: {rf_grid.best_params_}')
print(f'CV score: {rf_grid.best_score_:.4f} | Test: {accuracy_score(y_test, best_rf.predict(X_test)):.4f}')

Best params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
CV score: 0.8230 | Test: 0.9180


In [7]:
gb_grid = GridSearchCV(GradientBoostingClassifier(random_state=42),
    {'n_estimators':[100,200],'max_depth':[3,5],'learning_rate':[0.05,0.1],'subsample':[0.8,1.0]},
    cv=cv, scoring='accuracy', n_jobs=-1)
gb_grid.fit(X_train, y_train)
best_gb = gb_grid.best_estimator_
print(f'Best params: {gb_grid.best_params_}')
print(f'CV score: {gb_grid.best_score_:.4f} | Test: {accuracy_score(y_test, best_gb.predict(X_test)):.4f}')

Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
CV score: 0.7901 | Test: 0.8689


In [9]:
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

models = {'Logistic Regression':best_lr,'Random Forest':best_rf,'XGBoost (GBM)':best_gb}
results = {}
for name, m in models.items():
    p=m.predict(X_test); pr=m.predict_proba(X_test)[:,1]
    results[name]={'Accuracy':accuracy_score(y_test,p),'F1':f1_score(y_test,p),'ROC-AUC':roc_auc_score(y_test,pr)}

print(pd.DataFrame(results).T.round(4).to_string())

for fname, obj in [('logistic_model.pkl',best_lr),('rf_model.pkl',best_rf),('xgb_model.pkl',best_gb)]:
    pickle.dump(obj, open(f'models/{fname}','wb'))
print('\nAll models saved ✓')

                     Accuracy      F1  ROC-AUC
Logistic Regression    0.8525  0.8475   0.9556
Random Forest          0.9180  0.9153   0.9556
XGBoost (GBM)          0.8689  0.8667   0.9513

All models saved ✓
